In [ ]:
# Imports
import time

# Dependencies
import pandas as pd
import mwparserfromhell
from mwparserfromhell.nodes	import Template

# Library
from cards.classes import ScrapCard
from utils.utils import remove_from_list
from pipeline.builder import VanguardPipeline
from data.vanguard_data import VanguardStorage
from parsers.vanguard_parser import VanguardParser
from classifier.vanguard_classifier import VanguardClassifier
from classifier.classifier import process_items, sort_storage_list
from parsers.normalizers import normalize_length, raw_table_data_prepare
from api_builder.vanguard_api_build import MediaWikiAPI, VanguardScrapper

# Definition
JSONType = dict[str]
header = {
	"User-Agent": "VanguardScrapper/1.0 (Python; contact: kmarrero1993@gmail.com)"
}
first_param = {
	"action": "parse",
	"page": "List of Cardfight!! Vanguard Booster Sets",
	"format": "json"
}


In [ ]:
# Url Parser
web = MediaWikiAPI()
pipeline = VanguardPipeline(
	VanguardScrapper(web),
	VanguardParser(),
	VanguardClassifier(),
	VanguardStorage()
)
curl = pipeline.scrapper.api.get(params=first_param, headers=header)
crude_data = pipeline.scrapper.obtain_links(curl)
crude_consults = pipeline.parser.separate_urls(crude_data)
crude_consults = remove_from_list(crude_consults, [
	"Lyrical Monasterio", "The Mask Collection"
])
consults = pipeline.parser.make_consults("get", crude_consults)
crude_data = remove_from_list(crude_data, [
	"G Booster Set 8: Collezione del Combattente Vol.1",
	"G Booster Set 9: Collezione del Combattente Vol.2",
	"Thailand Booster Set 1: The Mask Collection",
	"G Booster Set 7: Giudizio delle Lame Splendenti",
	"G Booster Set 1: Trascendenza Interdimensionale"
])
process_items(crude_data, pipeline.classifier, pipeline.storage)
sort_storage_list(["LB", "G"], pipeline.classifier, pipeline.storage)
dz_consults = pipeline.parser.make_consults("consult", pipeline.storage.dz)
time.sleep(2)

curl = pipeline.scrapper.api.get(params=dz_consults[0], headers=header)
wikitex = pipeline.scrapper.obtain_wikitex(curl)

In [ ]:
lb_consults = pipeline.parser.make_consults("consult", pipeline.storage.lb)
curl = pipeline.scrapper.api.get(params=lb_consults[0], headers=header)
wikitex = pipeline.scrapper.obtain_wikitex(curl)
wikitex

In [ ]:
x = mwparserfromhell.parse(wikitex)
lst = []
for i in x.filter_templates():
	if ("CardList" in i.name):
		lst.append(i)
lst = remove_from_list(lst, ["{{CardList/header/D}}", "{{CardList/footer}}", "{{CardList/header}}"])
lst

In [ ]:
x = mwparserfromhell.parse(wikitex)

In [ ]:
d = pipeline.parser.infobox(x)

In [ ]:
rows = []
for i in range(0, len(lst)):
	x = normalize_length(lst[i].params, len(lst[i].params))
	x = raw_table_data_prepare(lst[i].params)
	row = ScrapCard(
		CardNo = x[0],
		Name = x[1],
		Grade = x[2],
		Race = x[3],
		Type = x[4],
		Rarity = x[5],
		Release= d["release date:"]
	)
	rows.append(row)
print(rows)

In [ ]:
data = [ScrapCard.model_dump() for ScrapCard in rows]

In [ ]:
db = pd.DataFrame(data, columns=["CardNo", "Name", "Grade", "Race", "Type", "Rarity", "Release"])

In [ ]:
db

In [ ]:
curl = pipeline.scrapper.api.get(params=dz_consults[0], headers=header)
content = pipeline.scrapper.obtain_links(curl)
pipeline.parser.clean_trash_from_set(crude_consults, content, 4)
card_consult = pipeline.parser.make_consults("consult", content)
main_table = pipeline.scrapper.api.get(params=card_consult[0], headers=header)